In [6]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Загрузка модели
with open('XGBoost_model.pkl', 'rb') as f:
    model = pickle.load(f)

# Загрузка данных
df = pd.read_csv("df_merged_cleaned_highest_corr.csv")
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')  # Важно: сортировка по дате!

# Подготовка фичей и целевой переменной
Xgb = df.drop(columns=['Цена на арматуру', 'Date', 'Лом_3А, РФ CPT ж/д Южный ФО, руб./т, без НДС', 
                      'Лом_3А, РФ CPT ж/д Центральный ФО, руб./т, без НДС', 
                      'Лом_3А, РФ FCA ж/д респ. Татарстан, руб./т, без НДС', 
                      'Лом_3А, РФ FCA ж/д Московский регион, руб./т, без НДС', 
                      'Лом_3А, РФ CPT ж/д Уральский ФО, руб./т, без НДС', 
                      'Чугун_CFR Турция, $/т'])
ygb = df['Цена на арматуру']

# Добавление скользящего среднего
Xgb['rolling_mean'] = ygb.rolling(window=3, min_periods=1).mean()

In [7]:
tscv = TimeSeriesSplit(n_splits=5)
metrics = {'R2': [], 'MAE': [], 'RMSE': []}

for train_index, test_index in tscv.split(Xgb):
    X_train, X_test = Xgb.iloc[train_index], Xgb.iloc[test_index]
    y_train, y_test = ygb.iloc[train_index], ygb.iloc[test_index]
    
    # Предсказание на тестовом наборе
    y_pred = model.predict(X_test)
    
    # Расчет метрик
    metrics['R2'].append(r2_score(y_test, y_pred))
    metrics['MAE'].append(mean_absolute_error(y_test, y_pred))
    metrics['RMSE'].append(np.sqrt(mean_squared_error(y_test, y_pred)))

# Средние метрики по фолдам
print(f"Средний R2: {np.mean(metrics['R2']):.2f}")
print(f"Средний MAE: {np.mean(metrics['MAE']):.2f}")
print(f"Средний RMSE: {np.mean(metrics['RMSE']):.2f}")
print(metrics['R2'])
print(metrics['MAE'])
print(metrics['RMSE'])

Средний R2: 0.97
Средний MAE: 918.09
Средний RMSE: 1378.81
[0.9688886220648072, 0.9715319575472519, 0.9795008158856213, 0.975006977136462, 0.9303742113420885]
[437.97826804577466, 394.8391285211268, 319.7781965228873, 1094.7168243838028, 2343.12802596831]
[658.5525776400572, 636.3748311412123, 469.8226252811685, 2109.911230933265, 3019.412412775288]
